In [2]:
import numpy as np
import pandas as pd
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences


In [3]:
df = pd.read_csv("train.csv",nrows=10000)
df['Review'] = df['Review'].astype(str)
df['Polarity'] = df['Polarity']-1
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df['cleaned'] = df['Review'].apply(clean_text)


vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df['cleaned'])
sequences = tokenizer.texts_to_sequences(df['cleaned'])

max_length = 100
X = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
y = df['Polarity'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

embedding_index = {}
with open("glove.6B.100d.txt", encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word, vector = values[0], np.asarray(values[1:], dtype='float32')
        embedding_index[word] = vector

embedding_dim = 100
word_index = tokenizer.word_index
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in word_index.items():
    if i < vocab_size:
        embedding_vector = embedding_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

In [4]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=vocab_size,
                              output_dim=embedding_dim,
                              weights=[embedding_matrix],
                              input_length=max_length,
                              trainable=False),
    tf.keras.layers.Bidirectional(tf.keras.layers.GRU(128, dropout=0.2, recurrent_dropout=0.2)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

2025-06-10 18:51:54.519444: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2025-06-10 18:51:54.519470: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2025-06-10 18:51:54.519474: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.00 GB
2025-06-10 18:51:54.519772: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-06-10 18:51:54.519787: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [5]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 100)          1000000   
                                                                 
 bidirectional (Bidirection  (None, 256)               176640    
 al)                                                             
                                                                 
 dense (Dense)               (None, 64)                16448     
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                                 
Total params: 1193153 (4.55 MB)
Trainable params: 193153 (754.50 KB)
Non-trainable params: 1000000 (3.81 MB)
_____________

In [ ]:
history = model.fit(X_train, y_train, validation_split=0.2, epochs=10,batch_size=256)

Epoch 1/10


2025-06-10 18:53:32.287829: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


 8/29 [=======>......................] - ETA: 7:10 - loss: 0.7125 - accuracy: 0.5010

In [ ]:
predictions = (model.predict(X_test) > 0.5).astype(int).flatten()
print("\nClassification Report:\n", classification_report(y_test, predictions))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, predictions))

In [ ]:
# Find me the sentences where there is the mismatch of predictions
mismatched_indices = np.where(predictions != y_test)[0]
mismatched_sentences = df.iloc[mismatched_indices]['Review'].values
print("\nMismatched Sentences:")
for sentence in mismatched_sentences:
    print(sentence)
# Save this to a csv file , with the columns 'Review', 'Predicted', 'Actual' and clip the review to 100 characters
mismatched_sentences = [sentence[:100] for sentence in mismatched_sentences]
mismatched_df = pd.DataFrame({
    'Review': mismatched_sentences,
    'Predicted': predictions[mismatched_indices],
    'Actual': y_test[mismatched_indices]
})
mismatched_df.to_csv("mismatched_predictions.csv", index=False)